# 📑Tutorial-3: Trace File Generation and Storage

**📜 traceset trace storage class Storer** 

NUSCAR defines the Storer class, which can quickly store side-channel traces and generate trace files. Combined with Container, it enables the processing and re-storage of trace files.

### Generating a Storage File

We will randomly generate some side-channel traces and store these traces in the test.zarr file.

In [1]:
import nuscar
import numpy as np

In [2]:
storer = nuscar.traceset.StorerZARR('test.zarr', mode='w') # mode = 'w' or 'a'; 'a' is append mode, which requires the meta data to remain consistent

In [3]:
# Randomly generate data and store it
for i in range(100):
    samples = np.random.random((20, 5000))
    p = np.random.randint(0, 256, size=(20, 16), dtype=np.uint8) 
    iv = np.random.randint(0, 256, size=(20, 16), dtype=np.uint8) # When storing, the first dimension is trace_id; the meta and samples must stay consistent across calls
    storer.update(samples, plaintext=p, iv=iv) # The meta in the storage file is 'plaintext' and 'iv'

storer.close() # Save the file

Inspect the stored trace file

In [4]:
reader = nuscar.traceset.ReaderZARR('test.zarr')
ctn = nuscar.traceset.ContainerZARR(reader)
ctn

Number of traces,Points per trace,meta information
2000,5000,"['plaintext', 'iv']"


Randomly generate some side-channel traces and append them to the test.zarr file.

In [5]:
storer = nuscar.traceset.StorerZARR('test.zarr', mode='a') # mode = 'w' or 'a'; 'a' is append mode, which requires the meta data to remain consistent

In [6]:
# Randomly generate data and store it
for i in range(10):
    samples = np.random.random((30, 5000))
    p = np.random.randint(0, 256, size=(30, 16), dtype=np.uint8) 
    iv = np.random.randint(0, 256, size=(30, 16), dtype=np.uint8) # In append mode, the meta data must stay consistent
    storer.update(samples, plaintext=p, iv=iv)

storer.close() # Save the file

Inspect the stored trace file

In [7]:
reader = nuscar.traceset.ReaderZARR('test.zarr')
ctn = nuscar.traceset.ContainerZARR(reader)
ctn

Number of traces,Points per trace,meta information
2300,5000,"['plaintext', 'iv']"


### Trace Re-storage

We can use the update_container function to re-store traces. By using the preprocessing function provided when creating the container, the traces can be preprocessed, for example denoising, filtering, resampling, etc.

In [8]:
reader = nuscar.traceset.ReaderZARR('test.zarr')

In [9]:
# Add a filtering operation when creating the container
def proc(s):
    shape = s.shape
    s = nuscar.signalproc.filter_lowpass(s, 1e8, 1e6).reshape(shape) # Keep the shape unchanged
    return s[:, 10:250] # Take the middle segment after filtering
ctn = nuscar.ContainerZARR(reader, func_preprocess=proc)

In [10]:
s = nuscar.StorerETS('test.ets') # Re-store in ETS format
s.update_container(ctn, metakeep=['plaintext']) # Keep only plaintext
s.close()

  0%|          | 0/2300 [00:00<?, ?it/s]

In [11]:
reader = nuscar.traceset.ReaderETS('test.ets')
ctn = nuscar.traceset.ContainerETS(reader)
ctn

Number of traces,Points per trace,meta information
2300,240,['plaintext']
